# Motion Blend Quality Analysis
## Quantitative and Qualitative Evaluation

**Experiments:**
1. `Punches_Air Kicking_fist_blend_0.50`
2. `Tai_Chi_Flow Yoga_Pose_Flow_blend_0.50`

**Methodology:** Following Tselepi et al. (2025) "Controllable Single-Shot Animation Blending with Temporal Conditioning"

**Pipeline:** Rydlr Moverse → Fivetran → Elasticsearch

**Hardware:** Intel/CUDA-compatible architecture

## 1. Setup and Dependencies

In [ ]:
# Import required libraries
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import torch
from pathlib import Path
import json
from datetime import datetime

# Set style for professional visualizations
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

# Add analysis module to path
sys.path.append('../analysis')
from analyse_blend import BVHMotionLoader, MotionMetricsCalculator, FivetranUploader

print("✅ Dependencies loaded")
print(f"🖥️  Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

## 2. Dataset Information

### Preprocessing Pipeline
- **Source:** Rydlr Moverse dataset
- **Format:** BVH (Biovision Hierarchy)
- **Sampling Rate:** 30 FPS
- **Frame Range:** 75-900 frames per sequence
- **Joints:** 24 key joints selected from 65-joint skeleton
- **Tracked Joints:** Pelvis, LeftWrist, RightWrist, LeftFoot, RightFoot

### Blend Configuration
- **Method:** Hierarchical SPADE-based modulation (PyTorch)
- **Weight:** 0.50 (equal blending)
- **Transition Window:** Frames 120-180 (2-second transition at 30 FPS)
- **Training:** ~3 hours per 360-frame sequence on NVIDIA RTX 4090

## 3. Metric Definitions

### L2 Velocity
Measures the difference in joint speed between consecutive frames:

$$\Delta v(t,j) = |v(t,j) - v(t-1,j)| \quad \text{where} \quad v(t,j) = ||\vec{v}(t,j)||_2$$

### L2 Acceleration  
Measures the temporal change in velocity:

$$\Delta\Delta v(t,j) = |\Delta v(t,j) - \Delta v(t-1,j)|$$

### Fréchet Inception Distance (FID)
Compares the distribution of generated motions to real motions (lower is better).

### Coverage (Cov)
Measures how well the generated motions cover the space of real motions (higher is better).

### Diversity Metrics
- **Global Diversity (GDiv):** Variance across entire sequence
- **Local Diversity (LDiv):** Average variance in sliding windows
- **Inter Diversity:** Variance between different joints
- **Intra Diversity:** Average variance within each joint trajectory

## 4. Experiment 1: Punches_Air × Kicking_fist

In [ ]:
# Load blend 1
blend1_name = "Punches_Air Kicking_fist_blend_0.50"
print(f"📂 Loading: {blend1_name}")

loader1 = BVHMotionLoader(f"../data/blends/{blend1_name}.bvh")
motion1_data = loader1.load()

print(f"✅ Loaded {motion1_data['frame_count']} frames")
print(f"   Duration: {motion1_data['duration']:.2f}s")
print(f"   Joints: {len(motion1_data['joint_names'])}")

In [ ]:
# Compute metrics
calculator1 = MotionMetricsCalculator(motion1_data, transition_window=(120, 180))
metrics1 = calculator1.compute_all_metrics()

print("\n📊 Metrics Summary:")
print(f"Transition Smoothness: {metrics1['transition_smoothness']:.4f}")
print(f"FID: {metrics1['fid']:.4f}")
print(f"Coverage: {metrics1['coverage']:.4f}")
print(f"Global Diversity: {metrics1['diversity']['global_diversity']:.4f}")

### Visualization: L2 Velocity and Acceleration

In [ ]:
# Compute velocity and acceleration for visualization
l2_vel1 = calculator1.compute_l2_velocity()
l2_acc1 = calculator1.compute_l2_acceleration(l2_vel1)

# Joint names and colors
joint_names = ['Pelvis', 'LeftWrist', 'RightWrist', 'LeftFoot', 'RightFoot']
colors = ['#3b82f6', '#f97316', '#84cc16', '#ef4444', '#a855f7']

# Create side-by-side plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# L2 Velocity
for i, (joint, color) in enumerate(zip(joint_names, colors)):
    ax1.plot(l2_vel1[:, i], label=joint, color=color, linewidth=2, alpha=0.8)

ax1.axvline(120, color='white', linestyle='--', alpha=0.5, linewidth=2, label='Transition Start')
ax1.axvline(180, color='white', linestyle='--', alpha=0.5, linewidth=2, label='Transition End')
ax1.axvspan(120, 180, alpha=0.1, color='cyan')
ax1.set_xlabel('Frame', fontsize=12)
ax1.set_ylabel('L2 Velocity Δv(t,j)', fontsize=12)
ax1.set_title('L2 Velocity - Joint Speed Difference', fontsize=14, fontweight='bold')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# L2 Acceleration
for i, (joint, color) in enumerate(zip(joint_names, colors)):
    ax2.plot(l2_acc1[:, i], label=joint, color=color, linewidth=2, alpha=0.8)

ax2.axvline(120, color='white', linestyle='--', alpha=0.5, linewidth=2)
ax2.axvline(180, color='white', linestyle='--', alpha=0.5, linewidth=2)
ax2.axvspan(120, 180, alpha=0.1, color='cyan')
ax2.set_xlabel('Frame', fontsize=12)
ax2.set_ylabel('L2 Acceleration ΔΔv(t,j)', fontsize=12)
ax2.set_title('L2 Acceleration - Temporal Change', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.suptitle(f'Motion Analysis: {blend1_name}', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'../outputs/analysis/{blend1_name}_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Visualization saved")

## 5. Experiment 2: Tai_Chi_Flow × Yoga_Pose_Flow

In [ ]:
# Load blend 2
blend2_name = "Tai_Chi_Flow Yoga_Pose_Flow_blend_0.50"
print(f"📂 Loading: {blend2_name}")

loader2 = BVHMotionLoader(f"../data/blends/{blend2_name}.bvh")
motion2_data = loader2.load()

print(f"✅ Loaded {motion2_data['frame_count']} frames")
print(f"   Duration: {motion2_data['duration']:.2f}s")

In [ ]:
# Compute metrics
calculator2 = MotionMetricsCalculator(motion2_data, transition_window=(120, 180))
metrics2 = calculator2.compute_all_metrics()

print("\n📊 Metrics Summary:")
print(f"Transition Smoothness: {metrics2['transition_smoothness']:.4f}")
print(f"FID: {metrics2['fid']:.4f}")
print(f"Coverage: {metrics2['coverage']:.4f}")
print(f"Global Diversity: {metrics2['diversity']['global_diversity']:.4f}")

In [ ]:
# Visualization for blend 2
l2_vel2 = calculator2.compute_l2_velocity()
l2_acc2 = calculator2.compute_l2_acceleration(l2_vel2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# L2 Velocity
for i, (joint, color) in enumerate(zip(joint_names, colors)):
    ax1.plot(l2_vel2[:, i], label=joint, color=color, linewidth=2, alpha=0.8)

ax1.axvline(120, color='white', linestyle='--', alpha=0.5, linewidth=2, label='Transition Start')
ax1.axvline(180, color='white', linestyle='--', alpha=0.5, linewidth=2, label='Transition End')
ax1.axvspan(120, 180, alpha=0.1, color='cyan')
ax1.set_xlabel('Frame', fontsize=12)
ax1.set_ylabel('L2 Velocity Δv(t,j)', fontsize=12)
ax1.set_title('L2 Velocity - Joint Speed Difference', fontsize=14, fontweight='bold')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# L2 Acceleration
for i, (joint, color) in enumerate(zip(joint_names, colors)):
    ax2.plot(l2_acc2[:, i], label=joint, color=color, linewidth=2, alpha=0.8)

ax2.axvline(120, color='white', linestyle='--', alpha=0.5, linewidth=2)
ax2.axvline(180, color='white', linestyle='--', alpha=0.5, linewidth=2)
ax2.axvspan(120, 180, alpha=0.1, color='cyan')
ax2.set_xlabel('Frame', fontsize=12)
ax2.set_ylabel('L2 Acceleration ΔΔv(t,j)', fontsize=12)
ax2.set_title('L2 Acceleration - Temporal Change', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.suptitle(f'Motion Analysis: {blend2_name}', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'../outputs/analysis/{blend2_name}_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Visualization saved")

## 6. Comparative Analysis

In [ ]:
# Create comparative dataframe
comparison_data = {
    'Blend': [blend1_name, blend2_name],
    'FID ↓': [metrics1['fid'], metrics2['fid']],
    'Cov ↑': [metrics1['coverage'], metrics2['coverage']],
    'GDiv ↑': [metrics1['diversity']['global_diversity'], metrics2['diversity']['global_diversity']],
    'LDiv ↑': [metrics1['diversity']['local_diversity'], metrics2['diversity']['local_diversity']],
    'InterDiv ↑': [metrics1['diversity']['inter_diversity'], metrics2['diversity']['inter_diversity']],
    'IntraDiv ↓': [metrics1['diversity']['intra_diversity'], metrics2['diversity']['intra_diversity']],
    'Smoothness': [metrics1['transition_smoothness'], metrics2['transition_smoothness']]
}

df_comparison = pd.DataFrame(comparison_data)
print("\n📊 COMPARATIVE METRICS TABLE")
print("="*100)
print(df_comparison.to_string(index=False))
print("="*100)

# Save to CSV
df_comparison.to_csv('../outputs/analysis/comparative_metrics.csv', index=False)
print("\n✅ Comparison table saved to CSV")

## 7. Upload to Fivetran → Elasticsearch

In [ ]:
# Upload metrics to pipeline
uploader = FivetranUploader(endpoint="http://localhost:5000/api/metrics")

print("📤 Uploading blend 1 metrics...")
success1 = uploader.upload_metrics(blend1_name, metrics1)

print("📤 Uploading blend 2 metrics...")
success2 = uploader.upload_metrics(blend2_name, metrics2)

if success1 and success2:
    print("\n✅ All metrics uploaded successfully!")
    print("   Ready for Elasticsearch indexing in moverse_blend_metrics table")
else:
    print("\n⚠️ Some uploads failed. Check logs.")

## 8. Analysis Summary

### Observed Smoothness
The transition smoothness scores indicate how well the two source motions blend together:
- **Higher scores** (closer to 1.0) indicate smoother transitions with minimal discontinuities
- **Lower scores** suggest visible artifacts or abrupt changes in the transition region

### Transition Realism
L2 velocity and acceleration metrics within the transition window (frames 120-180) should remain close to the overall average for realistic blends. Spikes in these metrics indicate discontinuities that may be perceptually jarring.

### Hardware Specifications
- **CPU:** Intel architecture (specify exact model)
- **GPU:** NVIDIA GeForce RTX 4090 (if available)
- **Training Time:** ~3 hours per 360-frame sequence
- **Inference:** Few seconds with skeleton ID maps

### References
1. **Perez et al. (2018)** - "FiLM: Visual Reasoning with a General Conditioning Layer" (ArXiv:1709.07871)
2. **Tselepi et al. (2025)** - "Controllable Single-Shot Animation Blending with Temporal Conditioning"
3. **Heusel et al. (2017)** - "GANs Trained by a Two Time-Scale Update Rule Converge to a Local Nash Equilibrium" (FID metric)

### Dataset Attribution
- Motion sequences from Rydlr Moverse pipeline
- Processed through Fivetran ingestion
- Indexed in Elasticsearch for searchability

## 9. Next Steps

1. **Expand Dataset:** Analyze additional blend combinations
2. **Perceptual Studies:** Conduct user studies to validate quantitative metrics
3. **Real-time Pipeline:** Integrate analysis into production blend generation
4. **Optimization:** Fine-tune blend weights based on metric feedback
5. **Visualization:** Generate 3D skeletal animations for qualitative review